# Phase 7 — FastAPI Prediction Service

- **7A.** Inspect artifacts and lock the API contract
- **7B.** Build schemas, configuration, and artifact repository
- **7C.** Create FastAPI application and endpoints
- **7D.** Add filters, freshness, readiness, and structured errors
- **7E.** Add focused API tests
- **7F.** Add Docker support and final validation


## **7A.** Inspect Phase 6 artifacts and define the public API contract

This inspection verifies:

- all required Phase 6 latest artifacts exist
- the Phase 6 validation status permits serving
- the forecast contains exactly 72 ordered hourly rows
- forecast timestamps and horizons are valid
- PM2.5, AQI, health-guidance, and alert fields are present
- the forecast and JSON artifacts belong to the same Phase 6 run
- Parquet and JSON values can be safely converted into API responses
- internal-only columns are separated from public API fields
- the initial endpoint inventory and response contract are documented


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor


### Locate the latest successful Phase 6 outputs

Phase 7 serves the contents of `aqi/latest`.

The `latest` directory is updated only after a successful Phase 6 run, while
immutable historical outputs remain under `aqi/runs/<run_id>`.

The API will use the latest directory by default, with its location becoming
environment-configurable in Phase 7B.

In [3]:
PHASE_6_LATEST_DIR = (
    PROJECT_ROOT
    / "aqi"
    / "latest"
)

PHASE_6_FORECAST_PATH = (
    PHASE_6_LATEST_DIR
    / "live_pm25_aqi_forecast.parquet"
)

PHASE_6_ALERT_EPISODES_PATH = (
    PHASE_6_LATEST_DIR
    / "alert_episodes.json"
)

PHASE_6_SUMMARY_PATH = (
    PHASE_6_LATEST_DIR
    / "aqi_forecast_summary.json"
)

PHASE_6_METADATA_PATH = (
    PHASE_6_LATEST_DIR
    / "aqi_metadata.json"
)

PHASE_6_VALIDATION_REPORT_PATH = (
    PHASE_6_LATEST_DIR
    / "phase_6_validation_report.json"
)

PHASE_6_PLOT_PATH = (
    PHASE_6_LATEST_DIR
    / "aqi_forecast_plot.png"
)

print("Phase 6 latest directory:", PHASE_6_LATEST_DIR)

Phase 6 latest directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/aqi/latest


In [4]:
required_phase_6_artifacts = {
    "AQI forecast": PHASE_6_FORECAST_PATH,
    "alert episodes": PHASE_6_ALERT_EPISODES_PATH,
    "forecast summary": PHASE_6_SUMMARY_PATH,
    "AQI metadata": PHASE_6_METADATA_PATH,
    "Phase 6 validation report": (
        PHASE_6_VALIDATION_REPORT_PATH
    ),
}

optional_phase_6_artifacts = {
    "AQI forecast plot": PHASE_6_PLOT_PATH,
}

phase_6_artifact_status_records = []

for artifact_name, artifact_path in {
    **required_phase_6_artifacts,
    **optional_phase_6_artifacts,
}.items():
    phase_6_artifact_status_records.append(
        {
            "artifact": artifact_name,
            "required": (
                artifact_name
                in required_phase_6_artifacts
            ),
            "exists": artifact_path.exists(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.exists()
                else None
            ),
            "path": str(artifact_path),
        }
    )

phase_6_artifact_status_df = pd.DataFrame(
    phase_6_artifact_status_records
)

display(phase_6_artifact_status_df)

,artifact,required,exists,size_bytes,path
0,AQI forecast,True,True,34860,/home/riyan/Riyan/projects/pearls-aqi-predicto...
1,alert episodes,True,True,2,/home/riyan/Riyan/projects/pearls-aqi-predicto...
2,forecast summary,True,True,808,/home/riyan/Riyan/projects/pearls-aqi-predicto...
3,AQI metadata,True,True,2349,/home/riyan/Riyan/projects/pearls-aqi-predicto...
4,Phase 6 validation report,True,True,982,/home/riyan/Riyan/projects/pearls-aqi-predicto...
5,AQI forecast plot,False,True,133667,/home/riyan/Riyan/projects/pearls-aqi-predicto...


In [5]:
missing_required_artifacts = (
    phase_6_artifact_status_df.loc[
        phase_6_artifact_status_df["required"]
        & ~phase_6_artifact_status_df["exists"],
        "artifact",
    ].tolist()
)

empty_required_artifacts = (
    phase_6_artifact_status_df.loc[
        phase_6_artifact_status_df["required"]
        & phase_6_artifact_status_df["exists"]
        & phase_6_artifact_status_df[
            "size_bytes"
        ].fillna(0).le(0),
        "artifact",
    ].tolist()
)

assert not missing_required_artifacts, (
    "Missing required Phase 6 artifacts: "
    f"{missing_required_artifacts}"
)

assert not empty_required_artifacts, (
    "Empty required Phase 6 artifacts: "
    f"{empty_required_artifacts}"
)

print("All required Phase 6 serving artifacts exist.")

All required Phase 6 serving artifacts exist.


### Load JSON artifacts safely

The API must detect malformed JSON instead of returning incomplete or misleading
responses.

This helper verifies that each JSON file contains the expected top-level type.

In [6]:
def load_json_artifact(
    path: Path,
    *,
    expected_type: type,
) -> Any:
    """Load JSON and validate its top-level structure."""

    if not path.exists():
        raise FileNotFoundError(
            f"JSON artifact was not found: {path}"
        )

    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file:
            payload = json.load(file)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Artifact contains invalid JSON: {path}"
        ) from exc

    if not isinstance(payload, expected_type):
        raise TypeError(
            "Unexpected JSON structure in "
            f"{path}. Expected "
            f"{expected_type.__name__}, received "
            f"{type(payload).__name__}."
        )

    return payload

### Load the latest AQI forecast package

The forecast is stored in Parquet to preserve datatypes and timestamps.

Summary, metadata, validation, and alert episodes are stored as JSON because
they represent smaller structured objects.

In [7]:
phase_6_forecast_df = pd.read_parquet(
    PHASE_6_FORECAST_PATH
)

phase_6_alert_episodes = load_json_artifact(
    PHASE_6_ALERT_EPISODES_PATH,
    expected_type=list,
)

phase_6_summary = load_json_artifact(
    PHASE_6_SUMMARY_PATH,
    expected_type=dict,
)

phase_6_metadata = load_json_artifact(
    PHASE_6_METADATA_PATH,
    expected_type=dict,
)

phase_6_validation_report = load_json_artifact(
    PHASE_6_VALIDATION_REPORT_PATH,
    expected_type=dict,
)

print(
    "Forecast shape:",
    phase_6_forecast_df.shape,
)

print(
    "Alert episode count:",
    len(phase_6_alert_episodes),
)

print(
    "Summary fields:",
    len(phase_6_summary),
)

print(
    "Metadata fields:",
    len(phase_6_metadata),
)

Forecast shape: (72, 45)
Alert episode count: 0
Summary fields: 20
Metadata fields: 7


### Inspect the enriched forecast schema

Phase 6 preserved the original Phase 5 forecast fields and added:

- indicative hourly AQI
- rolling 24-hour AQI
- AQI categories and colors
- alert basis and severity
- health messages
- recommended actions

The API will expose only the public-safe subset required by a dashboard or
consumer application.

In [8]:
forecast_schema_df = pd.DataFrame(
    {
        "column": phase_6_forecast_df.columns,
        "dtype": [
            str(dtype)
            for dtype
            in phase_6_forecast_df.dtypes
        ],
        "missing_values": [
            int(
                phase_6_forecast_df[
                    column
                ].isna().sum()
            )
            for column
            in phase_6_forecast_df.columns
        ],
    }
)

display(forecast_schema_df)

,column,dtype,missing_values
0,pipeline_run_id,str,0
1,prediction_generated_at_utc,"datetime64[us, UTC]",0
2,reference_time,"datetime64[us, UTC]",0
3,target_time,"datetime64[us, UTC]",0
4,forecast_horizon_hours,int64,0
5,predicted_pm25_ug_m3_raw,float64,0
6,predicted_pm25_ug_m3,float64,0
7,prediction_was_clipped,bool,0
8,prediction_source,string,0
9,location_name,str,0


In [9]:
display(
    phase_6_forecast_df.head(5)
)

display(
    phase_6_forecast_df.tail(5)
)

,pipeline_run_id,prediction_generated_at_utc,reference_time,target_time,forecast_horizon_hours,predicted_pm25_ug_m3_raw,predicted_pm25_ug_m3,prediction_was_clipped,prediction_source,location_name,...,alert_trigger_category,alert_level,alert_rank,alert_is_active,sensitive_groups_alert,general_population_alert,hazardous_alert,health_message,recommended_action,alert_message
0,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 05:00:00+00:00,1,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 61. Most people may conti...
1,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 06:00:00+00:00,2,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...
2,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 07:00:00+00:00,3,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...
3,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 08:00:00+00:00,4,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...
4,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 09:00:00+00:00,5,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...


,pipeline_run_id,prediction_generated_at_utc,reference_time,target_time,forecast_horizon_hours,predicted_pm25_ug_m3_raw,predicted_pm25_ug_m3,prediction_was_clipped,prediction_source,location_name,...,alert_trigger_category,alert_level,alert_rank,alert_is_active,sensitive_groups_alert,general_population_alert,hazardous_alert,health_message,recommended_action,alert_message
67,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-30 00:00:00+00:00,68,9.919868,9.919868,False,xgboost_shallower,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 53. Most people may conti...
68,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-30 01:00:00+00:00,69,9.919868,9.919868,False,xgboost_shallower,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 53. Most people may conti...
69,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-30 02:00:00+00:00,70,10.396403,10.396403,False,xgboost_shallower,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 53. Most people may conti...
70,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-30 03:00:00+00:00,71,10.672855,10.672855,False,xgboost_shallower,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 53. Most people may conti...
71,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-30 04:00:00+00:00,72,10.713322,10.713322,False,xgboost_shallower,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 53. Most people may conti...


### Classify internal and public forecast fields

Not every stored field should automatically become part of the public API.

Fields are grouped into:

- provenance and timestamps
- PM2.5 forecast values
- indicative hourly AQI
- rolling 24-hour AQI
- alert and health information
- internal implementation details

The API schema will be based on an intentional public contract rather than raw
Parquet rows.

In [10]:
FORECAST_IDENTITY_FIELDS = [
    "pipeline_run_id",
    "prediction_generated_at_utc",
    "reference_time",
    "target_time",
    "forecast_horizon_hours",
    "location_name",
]

PM25_PUBLIC_FIELDS = [
    "predicted_pm25_ug_m3",
]

INDICATIVE_AQI_PUBLIC_FIELDS = [
    "indicative_hourly_pm25_aqi",
    "indicative_hourly_aqi_category",
    "indicative_hourly_aqi_color_hex",
]

ROLLING_AQI_PUBLIC_FIELDS = [
    "rolling_24h_pm25_ug_m3",
    "rolling_24h_pm25_aqi",
    "rolling_24h_aqi_category",
    "rolling_24h_aqi_color_hex",
    "rolling_24h_pm25_is_complete",
    "rolling_24h_missing_hours",
    "rolling_observed_hour_count",
    "rolling_predicted_hour_count",
]

ALERT_PUBLIC_FIELDS = [
    "alert_level",
    "alert_basis",
    "alert_trigger_aqi",
    "alert_trigger_category",
    "alert_is_active",
    "sensitive_groups_alert",
    "general_population_alert",
    "hazardous_alert",
    "health_message",
    "recommended_action",
]

PUBLIC_HOURLY_FORECAST_FIELDS = [
    "target_time",
    "forecast_horizon_hours",
    *PM25_PUBLIC_FIELDS,
    *INDICATIVE_AQI_PUBLIC_FIELDS,
    *ROLLING_AQI_PUBLIC_FIELDS,
    *ALERT_PUBLIC_FIELDS,
]

public_field_contract_df = pd.DataFrame(
    [
        {
            "group": "identity",
            "field": field,
        }
        for field in FORECAST_IDENTITY_FIELDS
    ]
    + [
        {
            "group": "PM2.5",
            "field": field,
        }
        for field in PM25_PUBLIC_FIELDS
    ]
    + [
        {
            "group": "indicative hourly AQI",
            "field": field,
        }
        for field in INDICATIVE_AQI_PUBLIC_FIELDS
    ]
    + [
        {
            "group": "rolling 24-hour AQI",
            "field": field,
        }
        for field in ROLLING_AQI_PUBLIC_FIELDS
    ]
    + [
        {
            "group": "alerts and health",
            "field": field,
        }
        for field in ALERT_PUBLIC_FIELDS
    ]
)

display(public_field_contract_df)

,group,field
0,identity,pipeline_run_id
1,identity,prediction_generated_at_utc
2,identity,reference_time
3,identity,target_time
4,identity,forecast_horizon_hours
5,identity,location_name
6,PM2.5,predicted_pm25_ug_m3
7,indicative hourly AQI,indicative_hourly_pm25_aqi
8,indicative hourly AQI,indicative_hourly_aqi_category
9,indicative hourly AQI,indicative_hourly_aqi_color_hex


### Validate the public hourly forecast contract

Every field required by the proposed hourly API response must exist in the
latest Phase 6 forecast.

Missing rolling values may be valid in future runs when an exact 24-hour window
cannot be built. Therefore, rolling AQI fields will later be represented as
nullable Pydantic fields.

In [11]:
missing_public_forecast_fields = sorted(
    set(PUBLIC_HOURLY_FORECAST_FIELDS)
    .difference(
        phase_6_forecast_df.columns
    )
)

print(
    "Missing public forecast fields:",
    missing_public_forecast_fields,
)

assert not missing_public_forecast_fields

print(
    "All proposed public hourly fields are available."
)

Missing public forecast fields: []
All proposed public hourly fields are available.


### Validate timestamps and forecast order

All API timestamps must serialize as timezone-aware ISO 8601 UTC values.

The forecast must contain one reference timestamp and 72 unique hourly target
timestamps ordered by forecast horizon.

In [12]:
phase_6_forecast_df = (
    phase_6_forecast_df.copy()
)

timestamp_columns = [
    "prediction_generated_at_utc",
    "reference_time",
    "target_time",
]

for column in timestamp_columns:
    phase_6_forecast_df[column] = pd.to_datetime(
        phase_6_forecast_df[column],
        utc=True,
        errors="coerce",
    )

phase_6_forecast_df[
    "forecast_horizon_hours"
] = pd.to_numeric(
    phase_6_forecast_df[
        "forecast_horizon_hours"
    ],
    errors="coerce",
)

phase_6_forecast_df = (
    phase_6_forecast_df
    .sort_values("forecast_horizon_hours")
    .reset_index(drop=True)
)

In [13]:
forecast_contract_summary = {
    "rows": len(phase_6_forecast_df),
    "columns": len(
        phase_6_forecast_df.columns
    ),
    "first_horizon": (
        phase_6_forecast_df[
            "forecast_horizon_hours"
        ].min()
    ),
    "last_horizon": (
        phase_6_forecast_df[
            "forecast_horizon_hours"
        ].max()
    ),
    "unique_horizons": (
        phase_6_forecast_df[
            "forecast_horizon_hours"
        ].nunique()
    ),
    "unique_reference_times": (
        phase_6_forecast_df[
            "reference_time"
        ].nunique()
    ),
    "unique_target_times": (
        phase_6_forecast_df[
            "target_time"
        ].nunique()
    ),
    "duplicate_target_times": int(
        phase_6_forecast_df[
            "target_time"
        ].duplicated().sum()
    ),
    "missing_target_times": int(
        phase_6_forecast_df[
            "target_time"
        ].isna().sum()
    ),
}

display(
    pd.Series(
        forecast_contract_summary,
        name="value",
    ).to_frame()
)

,value
rows,72
columns,45
first_horizon,1
last_horizon,72
unique_horizons,72
unique_reference_times,1
unique_target_times,72
duplicate_target_times,0
missing_target_times,0


### Validate API-critical forecast values

The API must not serve a successful forecast containing invalid operational
PM2.5 values, missing indicative AQI values, missing alert levels, or invalid
horizons.

Rolling AQI values may be nullable only when the corresponding rolling window
is marked incomplete.

In [14]:
api_value_validation_summary = {
    "missing_predicted_pm25": int(
        phase_6_forecast_df[
            "predicted_pm25_ug_m3"
        ].isna().sum()
    ),
    "negative_predicted_pm25": int(
        phase_6_forecast_df[
            "predicted_pm25_ug_m3"
        ].lt(0).sum()
    ),
    "infinite_predicted_pm25": int(
        np.isinf(
            phase_6_forecast_df[
                "predicted_pm25_ug_m3"
            ].to_numpy(dtype=float)
        ).sum()
    ),
    "missing_indicative_aqi": int(
        phase_6_forecast_df[
            "indicative_hourly_pm25_aqi"
        ].isna().sum()
    ),
    "missing_indicative_category": int(
        phase_6_forecast_df[
            "indicative_hourly_aqi_category"
        ].isna().sum()
    ),
    "complete_rolling_windows": int(
        phase_6_forecast_df[
            "rolling_24h_pm25_is_complete"
        ].fillna(False).sum()
    ),
    "missing_rolling_aqi": int(
        phase_6_forecast_df[
            "rolling_24h_pm25_aqi"
        ].isna().sum()
    ),
    "missing_alert_levels": int(
        phase_6_forecast_df[
            "alert_level"
        ].isna().sum()
    ),
    "missing_health_messages": int(
        phase_6_forecast_df[
            "health_message"
        ].isna().sum()
    ),
    "missing_recommended_actions": int(
        phase_6_forecast_df[
            "recommended_action"
        ].isna().sum()
    ),
}

display(
    pd.Series(
        api_value_validation_summary,
        name="value",
    ).to_frame()
)

,value
missing_predicted_pm25,0
negative_predicted_pm25,0
infinite_predicted_pm25,0
missing_indicative_aqi,0
missing_indicative_category,0
complete_rolling_windows,72
missing_rolling_aqi,0
missing_alert_levels,0
missing_health_messages,0
missing_recommended_actions,0


In [15]:
complete_rolling_mask = (
    phase_6_forecast_df[
        "rolling_24h_pm25_is_complete"
    ]
    .fillna(False)
    .astype(bool)
)

complete_rolling_missing_values = int(
    phase_6_forecast_df.loc[
        complete_rolling_mask,
        "rolling_24h_pm25_aqi",
    ].isna().sum()
)

incomplete_rolling_non_null_values = int(
    phase_6_forecast_df.loc[
        ~complete_rolling_mask,
        "rolling_24h_pm25_aqi",
    ].notna().sum()
)

rolling_nullability_summary = {
    "complete_windows_with_missing_aqi": (
        complete_rolling_missing_values
    ),
    "incomplete_windows_with_non_null_aqi": (
        incomplete_rolling_non_null_values
    ),
}

display(
    pd.Series(
        rolling_nullability_summary,
        name="value",
    ).to_frame()
)

assert complete_rolling_missing_values == 0
assert incomplete_rolling_non_null_values == 0

print("Rolling AQI nullability rules validated.")

,value
complete_windows_with_missing_aqi,0
incomplete_windows_with_non_null_aqi,0


Rolling AQI nullability rules validated.


### Inspect summary, metadata, validation, and alert schemas

These artifacts will be returned by dedicated API endpoints.

The inspection confirms their current top-level keys before stable Pydantic
response models are created.

In [16]:
json_artifact_schema_df = pd.DataFrame(
    [
        {
            "artifact": "forecast summary",
            "top_level_type": type(
                phase_6_summary
            ).__name__,
            "record_count": len(
                phase_6_summary
            ),
            "top_level_fields": sorted(
                phase_6_summary.keys()
            ),
        },
        {
            "artifact": "AQI metadata",
            "top_level_type": type(
                phase_6_metadata
            ).__name__,
            "record_count": len(
                phase_6_metadata
            ),
            "top_level_fields": sorted(
                phase_6_metadata.keys()
            ),
        },
        {
            "artifact": "validation report",
            "top_level_type": type(
                phase_6_validation_report
            ).__name__,
            "record_count": len(
                phase_6_validation_report
            ),
            "top_level_fields": sorted(
                phase_6_validation_report.keys()
            ),
        },
        {
            "artifact": "alert episodes",
            "top_level_type": type(
                phase_6_alert_episodes
            ).__name__,
            "record_count": len(
                phase_6_alert_episodes
            ),
            "top_level_fields": (
                sorted(
                    phase_6_alert_episodes[
                        0
                    ].keys()
                )
                if phase_6_alert_episodes
                else []
            ),
        },
    ]
)

display(json_artifact_schema_df)

,artifact,top_level_type,record_count,top_level_fields
0,forecast summary,dict,20,"[active_alert_rows, alert_episode_count, forec..."
1,AQI metadata,dict,7,"[alert_policy, aqi_standard, breakpoints, gene..."
2,validation report,dict,6,"[checks, limitations, phase_6_run_id, source_p..."
3,alert episodes,list,0,[]


### Confirm Phase 6 serving approval

The FastAPI service may serve forecast data only when the Phase 6 validation
status is approved.

The API process itself will still be able to start when artifacts are invalid or
missing, but readiness and forecast endpoints must report the problem instead
of returning successful forecast data.

In [17]:
phase_6_status = str(
    phase_6_validation_report.get(
        "status",
        "",
    )
)

acceptable_phase_6_statuses = {
    "AQI_ALERT_PIPELINE_APPROVED",
    "AQI_ALERT_PIPELINE_APPROVED_WITH_LIMITATIONS",
}

phase_6_status_summary = {
    "status": phase_6_status,
    "acceptable_for_serving": (
        phase_6_status
        in acceptable_phase_6_statuses
    ),
    "phase_6_run_id": (
        phase_6_validation_report.get(
            "phase_6_run_id"
        )
    ),
    "source_phase_5_run_id": (
        phase_6_validation_report.get(
            "source_phase_5_run_id"
        )
    ),
}

display(
    pd.Series(
        phase_6_status_summary,
        name="value",
    ).to_frame()
)

assert (
    phase_6_status
    in acceptable_phase_6_statuses
)

print("Phase 6 status permits API serving.")

,value
status,AQI_ALERT_PIPELINE_APPROVED
acceptable_for_serving,True
phase_6_run_id,20260727T100418Z_aqi_ec8842b5
source_phase_5_run_id,20260727T064244Z_ec8842b5


Phase 6 status permits API serving.


### Confirm that all artifacts belong to the same run

Forecast, summary, metadata, and validation files must describe the same Phase 6
and source Phase 5 runs.

The future artifact repository will reject mismatched files instead of mixing
data from different pipeline executions.

In [18]:
forecast_phase_5_run_ids = (
    phase_6_forecast_df[
        "pipeline_run_id"
    ]
    .astype(str)
    .dropna()
    .unique()
    .tolist()
)

summary_phase_6_run_id = str(
    phase_6_summary.get(
        "phase_6_run_id"
    )
)

metadata_phase_6_run_id = str(
    phase_6_metadata.get(
        "phase_6_run_id"
    )
)

validation_phase_6_run_id = str(
    phase_6_validation_report.get(
        "phase_6_run_id"
    )
)

summary_source_phase_5_run_id = str(
    phase_6_summary.get(
        "source_phase_5_run_id"
    )
)

metadata_source_phase_5_run_id = str(
    phase_6_metadata.get(
        "source_phase_5_run_id"
    )
)

validation_source_phase_5_run_id = str(
    phase_6_validation_report.get(
        "source_phase_5_run_id"
    )
)

run_consistency_summary = {
    "forecast_phase_5_run_ids": (
        forecast_phase_5_run_ids
    ),
    "summary_phase_6_run_id": (
        summary_phase_6_run_id
    ),
    "metadata_phase_6_run_id": (
        metadata_phase_6_run_id
    ),
    "validation_phase_6_run_id": (
        validation_phase_6_run_id
    ),
    "summary_source_phase_5_run_id": (
        summary_source_phase_5_run_id
    ),
    "metadata_source_phase_5_run_id": (
        metadata_source_phase_5_run_id
    ),
    "validation_source_phase_5_run_id": (
        validation_source_phase_5_run_id
    ),
}

display(
    pd.Series(
        run_consistency_summary,
        name="value",
    ).to_frame()
)

,value
forecast_phase_5_run_ids,[20260727T064244Z_ec8842b5]
summary_phase_6_run_id,20260727T100418Z_aqi_ec8842b5
metadata_phase_6_run_id,20260727T100418Z_aqi_ec8842b5
validation_phase_6_run_id,20260727T100418Z_aqi_ec8842b5
summary_source_phase_5_run_id,20260727T064244Z_ec8842b5
metadata_source_phase_5_run_id,20260727T064244Z_ec8842b5
validation_source_phase_5_run_id,20260727T064244Z_ec8842b5


In [19]:
assert (
    summary_phase_6_run_id
    == metadata_phase_6_run_id
    == validation_phase_6_run_id
)

assert (
    summary_source_phase_5_run_id
    == metadata_source_phase_5_run_id
    == validation_source_phase_5_run_id
)

assert len(forecast_phase_5_run_ids) == 1

assert (
    forecast_phase_5_run_ids[0]
    == summary_source_phase_5_run_id
)

print("Phase 6 artifact run consistency validated.")

Phase 6 artifact run consistency validated.


### Inspect forecast generation time

The future API readiness endpoint will expose how old the latest Phase 6 output
is.

Phase 7B will centralize the actual staleness threshold in configuration.

This inspection calculates the current artifact age but does not reject the
historical development run.

In [20]:
phase_6_generated_at_utc = pd.to_datetime(
    phase_6_summary.get(
        "generated_at_utc"
    ),
    utc=True,
    errors="coerce",
)

current_utc_time = pd.Timestamp.now(
    tz="UTC"
)

if pd.isna(phase_6_generated_at_utc):
    forecast_age_hours = None
else:
    forecast_age_hours = (
        current_utc_time
        - phase_6_generated_at_utc
    ).total_seconds() / 3_600

freshness_inspection_summary = {
    "generated_at_utc": (
        phase_6_generated_at_utc
    ),
    "current_utc_time": current_utc_time,
    "forecast_age_hours": (
        forecast_age_hours
    ),
    "freshness_enforcement": (
        "Deferred to configured Phase 7 readiness logic"
    ),
}

display(
    pd.Series(
        freshness_inspection_summary,
        name="value",
    ).to_frame()
)

,value
generated_at_utc,2026-07-27 10:04:18.039784+00:00
current_utc_time,2026-07-27 15:46:52.353963+00:00
forecast_age_hours,5.709532
freshness_enforcement,Deferred to configured Phase 7 readiness logic


### Identify values requiring API serialization

Pandas and NumPy values cannot always be returned directly as JSON.

The artifact repository will normalize:

- pandas timestamps to timezone-aware Python datetimes
- NumPy numeric values to Python numbers
- pandas nullable integers and booleans
- `NaN`, `NaT`, and `pd.NA` to `None`

This inspection identifies the datatypes that the repository must handle.

In [21]:
serialization_type_summary_df = pd.DataFrame(
    [
        {
            "column": column,
            "pandas_dtype": str(
                phase_6_forecast_df[
                    column
                ].dtype
            ),
            "sample_python_type": (
                type(
                    phase_6_forecast_df[
                        column
                    ].dropna().iloc[0]
                ).__name__
                if phase_6_forecast_df[
                    column
                ].notna().any()
                else "no non-null values"
            ),
        }
        for column in (
            PUBLIC_HOURLY_FORECAST_FIELDS
        )
    ]
)

display(serialization_type_summary_df)

,column,pandas_dtype,sample_python_type
0,target_time,"datetime64[us, UTC]",Timestamp
1,forecast_horizon_hours,int64,int64
2,predicted_pm25_ug_m3,float64,float64
3,indicative_hourly_pm25_aqi,Int64,int64
4,indicative_hourly_aqi_category,str,str
5,indicative_hourly_aqi_color_hex,str,str
6,rolling_24h_pm25_ug_m3,float64,float64
7,rolling_24h_pm25_aqi,Int64,int64
8,rolling_24h_aqi_category,str,str
9,rolling_24h_aqi_color_hex,str,str


### Proposed Phase 7 endpoint inventory

The first FastAPI version will expose nine read-only endpoints.

### Health

- `GET /api/v1/health/live`
- `GET /api/v1/health/ready`

### Forecast

- `GET /api/v1/forecast`
- `GET /api/v1/forecast/hourly`
- `GET /api/v1/forecast/summary`

### Alerts

- `GET /api/v1/alerts`
- `GET /api/v1/alerts/active`

### Service information

- `GET /api/v1/metadata`
- `GET /api/v1/pipeline/status`

FastAPI will also provide:

- `/docs`
- `/redoc`
- `/openapi.json`

No pipeline-trigger endpoint will be exposed in the first implementation.

In [22]:
API_PREFIX = "/api/v1"

proposed_endpoint_inventory = [
    {
        "method": "GET",
        "path": f"{API_PREFIX}/health/live",
        "purpose": (
            "Confirm the API process is running."
        ),
        "requires_forecast": False,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/health/ready",
        "purpose": (
            "Report artifact readiness and freshness."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/forecast",
        "purpose": (
            "Return the complete forecast package."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/forecast/hourly",
        "purpose": (
            "Return filterable hourly forecast rows."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/forecast/summary",
        "purpose": (
            "Return the saved Phase 6 summary."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/alerts",
        "purpose": (
            "Return all forecast alert episodes."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/alerts/active",
        "purpose": (
            "Return current and upcoming episodes."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/metadata",
        "purpose": (
            "Return public-safe project and AQI metadata."
        ),
        "requires_forecast": True,
    },
    {
        "method": "GET",
        "path": f"{API_PREFIX}/pipeline/status",
        "purpose": (
            "Return Phase 5, Phase 6, row-count, "
            "freshness, and alert status."
        ),
        "requires_forecast": True,
    },
]

endpoint_inventory_df = pd.DataFrame(
    proposed_endpoint_inventory
)

display(endpoint_inventory_df)

,method,path,purpose,requires_forecast
0,GET,/api/v1/health/live,Confirm the API process is running.,False
1,GET,/api/v1/health/ready,Report artifact readiness and freshness.,True
2,GET,/api/v1/forecast,Return the complete forecast package.,True
3,GET,/api/v1/forecast/hourly,Return filterable hourly forecast rows.,True
4,GET,/api/v1/forecast/summary,Return the saved Phase 6 summary.,True
5,GET,/api/v1/alerts,Return all forecast alert episodes.,True
6,GET,/api/v1/alerts/active,Return current and upcoming episodes.,True
7,GET,/api/v1/metadata,Return public-safe project and AQI metadata.,True
8,GET,/api/v1/pipeline/status,"Return Phase 5, Phase 6, row-count, freshness,...",True


### Public hourly forecast record

Each hourly response will intentionally expose only stable, useful fields.

Internal values such as raw model predictions, clipping flags, serialized model
details, local paths, and intermediate severity columns will remain private.

Rolling 24-hour fields will be nullable when an exact trailing window is
unavailable.

In [23]:
public_hourly_response_contract = [
    {
        "public_field": "target_time_utc",
        "artifact_field": "target_time",
        "nullable": False,
    },
    {
        "public_field": "forecast_horizon_hours",
        "artifact_field": "forecast_horizon_hours",
        "nullable": False,
    },
    {
        "public_field": "predicted_pm25_ug_m3",
        "artifact_field": "predicted_pm25_ug_m3",
        "nullable": False,
    },
    {
        "public_field": "indicative_hourly_pm25_aqi",
        "artifact_field": "indicative_hourly_pm25_aqi",
        "nullable": False,
    },
    {
        "public_field": "indicative_hourly_aqi_category",
        "artifact_field": "indicative_hourly_aqi_category",
        "nullable": False,
    },
    {
        "public_field": "indicative_hourly_aqi_color_hex",
        "artifact_field": "indicative_hourly_aqi_color_hex",
        "nullable": False,
    },
    {
        "public_field": "rolling_24h_pm25_ug_m3",
        "artifact_field": "rolling_24h_pm25_ug_m3",
        "nullable": True,
    },
    {
        "public_field": "rolling_24h_pm25_aqi",
        "artifact_field": "rolling_24h_pm25_aqi",
        "nullable": True,
    },
    {
        "public_field": "rolling_24h_aqi_category",
        "artifact_field": "rolling_24h_aqi_category",
        "nullable": True,
    },
    {
        "public_field": "rolling_24h_aqi_color_hex",
        "artifact_field": "rolling_24h_aqi_color_hex",
        "nullable": True,
    },
    {
        "public_field": "rolling_24h_pm25_is_complete",
        "artifact_field": "rolling_24h_pm25_is_complete",
        "nullable": False,
    },
    {
        "public_field": "alert_level",
        "artifact_field": "alert_level",
        "nullable": False,
    },
    {
        "public_field": "alert_basis",
        "artifact_field": "alert_basis",
        "nullable": False,
    },
    {
        "public_field": "alert_is_active",
        "artifact_field": "alert_is_active",
        "nullable": False,
    },
    {
        "public_field": "sensitive_groups_alert",
        "artifact_field": "sensitive_groups_alert",
        "nullable": False,
    },
    {
        "public_field": "general_population_alert",
        "artifact_field": "general_population_alert",
        "nullable": False,
    },
    {
        "public_field": "hazardous_alert",
        "artifact_field": "hazardous_alert",
        "nullable": False,
    },
    {
        "public_field": "health_message",
        "artifact_field": "health_message",
        "nullable": False,
    },
    {
        "public_field": "recommended_action",
        "artifact_field": "recommended_action",
        "nullable": False,
    },
]

public_hourly_response_contract_df = (
    pd.DataFrame(
        public_hourly_response_contract
    )
)

display(
    public_hourly_response_contract_df
)

,public_field,artifact_field,nullable
0,target_time_utc,target_time,False
1,forecast_horizon_hours,forecast_horizon_hours,False
2,predicted_pm25_ug_m3,predicted_pm25_ug_m3,False
3,indicative_hourly_pm25_aqi,indicative_hourly_pm25_aqi,False
4,indicative_hourly_aqi_category,indicative_hourly_aqi_category,False
5,indicative_hourly_aqi_color_hex,indicative_hourly_aqi_color_hex,False
6,rolling_24h_pm25_ug_m3,rolling_24h_pm25_ug_m3,True
7,rolling_24h_pm25_aqi,rolling_24h_pm25_aqi,True
8,rolling_24h_aqi_category,rolling_24h_aqi_category,True
9,rolling_24h_aqi_color_hex,rolling_24h_aqi_color_hex,True


### Hourly forecast filters

The first API version will support a compact set of useful filters:

- minimum forecast horizon
- maximum forecast horizon
- AQI category
- alert level
- active alerts only

Timestamp filters are not necessary initially because this project always
returns one fixed 72-hour hourly forecast and horizon filtering is simpler for
dashboard consumers.

In [24]:
hourly_filter_contract_df = pd.DataFrame(
    [
        {
            "parameter": "minimum_horizon",
            "type": "integer",
            "allowed_values": "1 through 72",
            "required": False,
        },
        {
            "parameter": "maximum_horizon",
            "type": "integer",
            "allowed_values": "1 through 72",
            "required": False,
        },
        {
            "parameter": "category",
            "type": "AQI category enum",
            "allowed_values": (
                "Good, Moderate, Unhealthy for "
                "Sensitive Groups, Unhealthy, "
                "Very Unhealthy, Hazardous"
            ),
            "required": False,
        },
        {
            "parameter": "alert_level",
            "type": "alert-level enum",
            "allowed_values": (
                "NORMAL, ADVISORY, WARNING, "
                "SEVERE, EMERGENCY"
            ),
            "required": False,
        },
        {
            "parameter": "alerts_only",
            "type": "boolean",
            "allowed_values": "true or false",
            "required": False,
        },
    ]
)

display(hourly_filter_contract_df)

,parameter,type,allowed_values,required
0,minimum_horizon,integer,1 through 72,False
1,maximum_horizon,integer,1 through 72,False
2,category,AQI category enum,"Good, Moderate, Unhealthy for Sensitive Groups...",False
3,alert_level,alert-level enum,"NORMAL, ADVISORY, WARNING, SEVERE, EMERGENCY",False
4,alerts_only,boolean,true or false,False


### Private service information

The API must not expose secrets, machine-specific paths, full environment
variables, stack traces, serialized model files, or raw provider responses.

The following checks provide an initial list of private fields and keywords that
will later be enforced through Pydantic schemas and tests.

In [25]:
PRIVATE_ARTIFACT_FIELDS = {
    "predicted_pm25_ug_m3_raw",
    "prediction_was_clipped",
    "selected_strategy",
    "sensor_id",
    "indicative_hourly_aqi_severity_rank",
    "rolling_24h_aqi_severity_rank",
    "alert_rank",
}

PRIVATE_RESPONSE_KEYWORDS = {
    "api_key",
    "secret",
    "password",
    "local_path",
    "filesystem_path",
    "model_path",
    "artifact_path",
    "stack_trace",
    "traceback",
}

private_field_inspection_df = pd.DataFrame(
    {
        "private_artifact_field": sorted(
            PRIVATE_ARTIFACT_FIELDS
        )
    }
)

display(private_field_inspection_df)

,private_artifact_field
0,alert_rank
1,indicative_hourly_aqi_severity_rank
2,predicted_pm25_ug_m3_raw
3,prediction_was_clipped
4,rolling_24h_aqi_severity_rank
5,selected_strategy
6,sensor_id


## **7B.** Configuration, schemas, and artifact repository

This subphase creates the reusable foundation for the FastAPI service.

It introduces:

- environment-based API configuration
- stable AQI, alert, freshness, and readiness enums
- explicit Pydantic response schemas
- validated Phase 6 artifact loading
- artifact run-consistency checks
- JSON-safe conversion of pandas and NumPy values
- a small thread-safe in-memory cache
- configurable forecast freshness classification

No routes or FastAPI application are created yet.

The artifact repository is the only future API component allowed to read the
Phase 6 Parquet and JSON files directly.

In [26]:
import importlib

import app.api.config
import app.api.schemas
import app.api.services.artifact_repository

importlib.reload(app.api.config)
importlib.reload(app.api.schemas)
importlib.reload(
    app.api.services.artifact_repository
)

from app.api.config import (
    APISettings,
    get_api_settings,
)

from app.api.schemas import (
    AQICategory,
    AlertLevel,
    FreshnessStatus,
    HourlyForecastRecord,
    ReadinessStatus,
)

from app.api.services.artifact_repository import (
    ArtifactRepository,
    dataframe_to_public_records,
    json_safe_value,
)

### Validate environment-based configuration

The default configuration should resolve the existing `aqi/latest` directory
and expose non-secret application, cache, freshness, and CORS settings.

Environment variables can override these values without modifying source code.

In [27]:
settings = get_api_settings()

settings_summary = {
    "application_name": (
        settings.application_name
    ),
    "application_version": (
        settings.application_version
    ),
    "environment": settings.environment,
    "api_prefix": settings.api_prefix,
    "host": settings.host,
    "port": settings.port,
    "allowed_cors_origins": (
        settings.allowed_cors_origins
    ),
    "phase_6_latest_directory": str(
        settings.phase_6_latest_directory
    ),
    "artifact_cache_seconds": (
        settings.artifact_cache_seconds
    ),
    "forecast_aging_threshold_hours": (
        settings.forecast_aging_threshold_hours
    ),
    "forecast_staleness_threshold_hours": (
        settings
        .forecast_staleness_threshold_hours
    ),
}

display(
    pd.Series(
        settings_summary,
        name="value",
    ).to_frame()
)

,value
application_name,Pearls AQI Predictor API
application_version,1.0.0
environment,development
api_prefix,/api/v1
host,0.0.0.0
port,8000
allowed_cors_origins,"(http://localhost:3000, http://localhost:8501)"
phase_6_latest_directory,/home/riyan/Riyan/projects/pearls-aqi-predicto...
artifact_cache_seconds,60
forecast_aging_threshold_hours,6.0


In [28]:
configured_artifact_paths = {
    "forecast": settings.forecast_path,
    "alert episodes": (
        settings.alert_episodes_path
    ),
    "summary": settings.summary_path,
    "metadata": settings.metadata_path,
    "validation report": (
        settings.validation_report_path
    ),
}

configured_path_status_df = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": artifact_path.exists(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.exists()
                else None
            ),
            "filename": artifact_path.name,
        }
        for artifact_name, artifact_path
        in configured_artifact_paths.items()
    ]
)

display(configured_path_status_df)

assert configured_path_status_df[
    "exists"
].all()

,artifact,exists,size_bytes,filename
0,forecast,True,34860,live_pm25_aqi_forecast.parquet
1,alert episodes,True,2,alert_episodes.json
2,summary,True,808,aqi_forecast_summary.json
3,metadata,True,2349,aqi_metadata.json
4,validation report,True,982,phase_6_validation_report.json


### Load the latest Phase 6 package through the repository

This is the same path future routes will use.

The repository validates all files as one consistent package before returning
forecast data.

In [29]:
artifact_repository = ArtifactRepository(
    settings
)

phase_7_artifact_bundle = (
    artifact_repository.load_latest(
        force_reload=True
    )
)

repository_load_summary = {
    "phase_6_run_id": (
        phase_7_artifact_bundle
        .phase_6_run_id
    ),
    "source_phase_5_run_id": (
        phase_7_artifact_bundle
        .source_phase_5_run_id
    ),
    "forecast_rows": len(
        phase_7_artifact_bundle
        .forecast_df
    ),
    "forecast_columns": len(
        phase_7_artifact_bundle
        .forecast_df.columns
    ),
    "alert_episode_count": len(
        phase_7_artifact_bundle
        .alert_episodes
    ),
    "generated_at_utc": (
        phase_7_artifact_bundle
        .generated_at_utc
    ),
    "loaded_at_utc": (
        phase_7_artifact_bundle
        .loaded_at_utc
    ),
    "freshness_status": (
        phase_7_artifact_bundle
        .freshness.status
    ),
    "age_hours": (
        phase_7_artifact_bundle
        .freshness.age_hours
    ),
}

display(
    pd.Series(
        repository_load_summary,
        name="value",
    ).to_frame()
)

,value
phase_6_run_id,20260727T100418Z_aqi_ec8842b5
source_phase_5_run_id,20260727T064244Z_ec8842b5
forecast_rows,72
forecast_columns,45
alert_episode_count,0
generated_at_utc,2026-07-27 10:04:18.039784+00:00
loaded_at_utc,2026-07-27 15:46:53.731417+00:00
freshness_status,FreshnessStatus.FRESH
age_hours,5.71


### Confirm in-memory cache reuse

Repeated requests should not reread and revalidate all artifacts while the
cache is valid and source files are unchanged.

The same immutable artifact-bundle object should be returned from the cache.

In [30]:
first_cached_bundle = (
    artifact_repository.load_latest()
)

second_cached_bundle = (
    artifact_repository.load_latest()
)

cache_reused = (
    first_cached_bundle
    is second_cached_bundle
)

print("Cache reused:", cache_reused)

assert cache_reused

Cache reused: True


### Confirm explicit cache refresh

A forced reload should reread and revalidate the artifacts, producing a new
bundle object while preserving the same pipeline identity.

In [31]:
reloaded_bundle = (
    artifact_repository.load_latest(
        force_reload=True
    )
)

forced_reload_summary = {
    "new_bundle_object": (
        reloaded_bundle
        is not second_cached_bundle
    ),
    "same_phase_6_run_id": (
        reloaded_bundle.phase_6_run_id
        == second_cached_bundle.phase_6_run_id
    ),
    "same_phase_5_run_id": (
        reloaded_bundle
        .source_phase_5_run_id
        == second_cached_bundle
        .source_phase_5_run_id
    ),
}

display(
    pd.Series(
        forced_reload_summary,
        name="value",
    ).to_frame()
)

assert forced_reload_summary[
    "new_bundle_object"
]

assert forced_reload_summary[
    "same_phase_6_run_id"
]

,value
new_bundle_object,True
same_phase_6_run_id,True
same_phase_5_run_id,True


### Validate configurable freshness behavior

Freshness is calculated independently from artifact validity.

The repository supports:

- `FRESH` before the aging threshold
- `AGING` between the aging and staleness thresholds
- `STALE` at or after the staleness threshold

In [32]:
from datetime import (
    datetime,
    timedelta,
    timezone,
)


freshness_now = datetime(
    2026,
    1,
    1,
    12,
    0,
    tzinfo=timezone.utc,
)

fresh_result = (
    artifact_repository.calculate_freshness(
        freshness_now
        - timedelta(hours=1),
        now_utc=freshness_now,
    )
)

aging_result = (
    artifact_repository.calculate_freshness(
        freshness_now
        - timedelta(hours=8),
        now_utc=freshness_now,
    )
)

stale_result = (
    artifact_repository.calculate_freshness(
        freshness_now
        - timedelta(hours=13),
        now_utc=freshness_now,
    )
)

freshness_test_df = pd.DataFrame(
    [
        {
            "case": "fresh",
            "age_hours": fresh_result.age_hours,
            "status": fresh_result.status,
        },
        {
            "case": "aging",
            "age_hours": aging_result.age_hours,
            "status": aging_result.status,
        },
        {
            "case": "stale",
            "age_hours": stale_result.age_hours,
            "status": stale_result.status,
        },
    ]
)

display(freshness_test_df)

assert (
    fresh_result.status
    == FreshnessStatus.FRESH
)

assert (
    aging_result.status
    == FreshnessStatus.AGING
)

assert (
    stale_result.status
    == FreshnessStatus.STALE
)

,case,age_hours,status
0,fresh,1.0,FRESH
1,aging,8.0,AGING
2,stale,13.0,STALE


### Validate API-safe serialization

The repository must convert pandas and NumPy values into standard Python values.

Missing and non-finite values must become `None`, ensuring API JSON never
contains invalid `NaN` or infinity values.

In [33]:
serialization_examples = {
    "pandas_timestamp": pd.Timestamp(
        "2026-01-01 00:00:00+00:00"
    ),
    "numpy_integer": np.int64(72),
    "numpy_float": np.float64(14.3),
    "numpy_boolean": np.bool_(True),
    "numpy_nan": np.float64(np.nan),
    "pandas_na": pd.NA,
    "nested_values": [
        np.int64(1),
        np.float64(np.nan),
    ],
}

safe_serialization_examples = (
    json_safe_value(
        serialization_examples
    )
)

serialization_validation_df = pd.DataFrame(
    [
        {
            "field": key,
            "safe_value": value,
            "safe_type": type(
                value
            ).__name__,
        }
        for key, value
        in safe_serialization_examples.items()
    ]
)

display(serialization_validation_df)

assert (
    safe_serialization_examples[
        "numpy_integer"
    ]
    == 72
)

assert (
    safe_serialization_examples[
        "numpy_nan"
    ]
    is None
)

assert (
    safe_serialization_examples[
        "pandas_na"
    ]
    is None
)

assert (
    safe_serialization_examples[
        "nested_values"
    ][1]
    is None
)

,field,safe_value,safe_type
0,pandas_timestamp,2026-01-01 00:00:00+00:00,datetime
1,numpy_integer,72,int
2,numpy_float,14.3,float
3,numpy_boolean,True,bool
4,numpy_nan,None,NoneType
5,pandas_na,None,NoneType
6,nested_values,"[1, None]",list


### Convert DataFrame rows without exposing raw pandas values

The repository provides a reusable record conversion helper.

The route layer will later select and rename public fields before validating
them with Pydantic.

In [34]:
safe_forecast_records = (
    dataframe_to_public_records(
        phase_7_artifact_bundle
        .forecast_df.head(3)
    )
)

print(
    "Converted record count:",
    len(safe_forecast_records),
)

display(
    pd.DataFrame(
        safe_forecast_records
    )
)

assert len(
    safe_forecast_records
) == 3

assert isinstance(
    safe_forecast_records[0],
    dict,
)

Converted record count: 3


,pipeline_run_id,prediction_generated_at_utc,reference_time,target_time,forecast_horizon_hours,predicted_pm25_ug_m3_raw,predicted_pm25_ug_m3,prediction_was_clipped,prediction_source,location_name,...,alert_trigger_category,alert_level,alert_rank,alert_is_active,sensitive_groups_alert,general_population_alert,hazardous_alert,health_message,recommended_action,alert_message
0,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 05:00:00+00:00,1,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 61. Most people may conti...
1,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 06:00:00+00:00,2,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...
2,20260727T064244Z_ec8842b5,2026-07-27 06:42:44.492588+00:00,2026-07-27 04:00:00+00:00,2026-07-27 07:00:00+00:00,3,14.3,14.3,False,current_pm25_persistence,Zafar Memon DHA,...,Moderate,NORMAL,0,False,False,False,False,"Air quality is generally acceptable, although ...",Most people may continue normal activities. Un...,NORMAL: Moderate AQI 62. Most people may conti...


### Validate the proposed public hourly schema

One actual Phase 6 row is mapped to the stable public field names and validated
through Pydantic.

This confirms that the artifact schema and public API schema are compatible.

In [35]:
first_forecast_row = (
    phase_7_artifact_bundle
    .forecast_df.iloc[0]
)

hourly_record_payload = {
    "target_time_utc": (
        first_forecast_row[
            "target_time"
        ]
    ),
    "forecast_horizon_hours": int(
        first_forecast_row[
            "forecast_horizon_hours"
        ]
    ),
    "predicted_pm25_ug_m3": float(
        first_forecast_row[
            "predicted_pm25_ug_m3"
        ]
    ),
    "indicative_hourly_pm25_aqi": int(
        first_forecast_row[
            "indicative_hourly_pm25_aqi"
        ]
    ),
    "indicative_hourly_aqi_category": (
        first_forecast_row[
            "indicative_hourly_aqi_category"
        ]
    ),
    "indicative_hourly_aqi_color_hex": (
        first_forecast_row[
            "indicative_hourly_aqi_color_hex"
        ]
    ),
    "rolling_24h_pm25_ug_m3": float(
        first_forecast_row[
            "rolling_24h_pm25_ug_m3"
        ]
    ),
    "rolling_24h_pm25_aqi": int(
        first_forecast_row[
            "rolling_24h_pm25_aqi"
        ]
    ),
    "rolling_24h_aqi_category": (
        first_forecast_row[
            "rolling_24h_aqi_category"
        ]
    ),
    "rolling_24h_aqi_color_hex": (
        first_forecast_row[
            "rolling_24h_aqi_color_hex"
        ]
    ),
    "rolling_24h_pm25_is_complete": bool(
        first_forecast_row[
            "rolling_24h_pm25_is_complete"
        ]
    ),
    "rolling_24h_missing_hours": int(
        first_forecast_row[
            "rolling_24h_missing_hours"
        ]
    ),
    "rolling_observed_hour_count": int(
        first_forecast_row[
            "rolling_observed_hour_count"
        ]
    ),
    "rolling_predicted_hour_count": int(
        first_forecast_row[
            "rolling_predicted_hour_count"
        ]
    ),
    "alert_level": (
        first_forecast_row[
            "alert_level"
        ]
    ),
    "alert_basis": (
        first_forecast_row[
            "alert_basis"
        ]
    ),
    "alert_trigger_aqi": int(
        first_forecast_row[
            "alert_trigger_aqi"
        ]
    ),
    "alert_trigger_category": (
        first_forecast_row[
            "alert_trigger_category"
        ]
    ),
    "alert_is_active": bool(
        first_forecast_row[
            "alert_is_active"
        ]
    ),
    "sensitive_groups_alert": bool(
        first_forecast_row[
            "sensitive_groups_alert"
        ]
    ),
    "general_population_alert": bool(
        first_forecast_row[
            "general_population_alert"
        ]
    ),
    "hazardous_alert": bool(
        first_forecast_row[
            "hazardous_alert"
        ]
    ),
    "health_message": (
        first_forecast_row[
            "health_message"
        ]
    ),
    "recommended_action": (
        first_forecast_row[
            "recommended_action"
        ]
    ),
}

validated_hourly_record = (
    HourlyForecastRecord.model_validate(
        hourly_record_payload
    )
)

display(
    pd.Series(
        validated_hourly_record.model_dump(),
        name="value",
    ).to_frame()
)

print(
    "Public hourly schema validation passed."
)

,value
target_time_utc,2026-07-27 05:00:00+00:00
forecast_horizon_hours,1
predicted_pm25_ug_m3,14.3
indicative_hourly_pm25_aqi,61
indicative_hourly_aqi_category,Moderate
indicative_hourly_aqi_color_hex,#FFFF00
rolling_24h_pm25_ug_m3,14.795833
rolling_24h_pm25_aqi,61
rolling_24h_aqi_category,Moderate
rolling_24h_aqi_color_hex,#FFFF00


Public hourly schema validation passed.


## **7C.** FastAPI application and read-only endpoints

The FastAPI service now exposes the latest validated Phase 6 output.

The application includes:

- lifespan-based artifact-cache initialization
- dependency injection for the shared repository
- request ID and duration middleware
- configured CORS
- gzip support for larger responses
- structured exception handling
- liveness and readiness checks
- complete and filterable forecast endpoints
- summary and alert endpoints
- public metadata and pipeline-status endpoints
- automatic OpenAPI documentation

Normal GET requests read only the latest validated artifact package. They do
not call external data providers or run the model.

In [36]:
import importlib

import app.api.main

importlib.reload(app.api.main)

from app.api.main import app

print("Application title:", app.title)
print("Application version:", app.version)
print("OpenAPI path:", app.openapi_url)

Application title: Pearls AQI Predictor API
Application version: 1.0.0
OpenAPI path: /openapi.json


In [37]:
openapi_schema = app.openapi()

registered_routes = []

for path, path_item in openapi_schema["paths"].items():
    for method, operation in path_item.items():
        if method.lower() not in {
            "get",
            "post",
            "put",
            "patch",
            "delete",
            "options",
            "head",
            "trace",
        }:
            continue

        registered_routes.append(
            {
                "path": path,
                "method": method.upper(),
                "name": operation.get(
                    "summary"
                ),
                "operation_id": operation.get(
                    "operationId"
                ),
                "tags": operation.get(
                    "tags",
                    [],
                ),
            }
        )

registered_routes_df = (
    pd.DataFrame(registered_routes)
    .sort_values(
        ["path", "method"]
    )
    .reset_index(drop=True)
)

display(registered_routes_df)

,path,method,name,operation_id,tags
0,/api/v1/alerts,GET,Get all forecast alert episodes,get_alert_episodes_api_v1_alerts_get,[Alerts]
1,/api/v1/alerts/active,GET,Get current and upcoming alert episodes,get_active_alerts_api_v1_alerts_active_get,[Alerts]
2,/api/v1/forecast,GET,Get the complete 72-hour forecast,get_complete_forecast_api_v1_forecast_get,[Forecast]
3,/api/v1/forecast/hourly,GET,Get filterable hourly forecasts,get_hourly_forecast_api_v1_forecast_hourly_get,[Forecast]
4,/api/v1/forecast/summary,GET,Get the forecast summary,get_forecast_summary_api_v1_forecast_summary_get,[Forecast]
5,/api/v1/health/live,GET,Check service liveness,get_liveness_api_v1_health_live_get,[Health]
6,/api/v1/health/ready,GET,Check forecast readiness,get_readiness_api_v1_health_ready_get,[Health]
7,/api/v1/metadata,GET,Get public project metadata,get_metadata_api_v1_metadata_get,[Metadata and operations]
8,/api/v1/pipeline/status,GET,Get the latest pipeline status,get_pipeline_status_api_v1_pipeline_status_get,[Metadata and operations]


### Exercise the API with FastAPI TestClient

Using `TestClient` runs the lifespan handler, initializes the repository, and
allows the endpoints to be validated without starting a separate Uvicorn
process.

In [38]:
from fastapi.testclient import TestClient


with TestClient(app) as client:
    live_response = client.get(
        "/api/v1/health/live"
    )

    ready_response = client.get(
        "/api/v1/health/ready"
    )

    forecast_response = client.get(
        "/api/v1/forecast"
    )

    hourly_response = client.get(
        (
            "/api/v1/forecast/hourly"
            "?minimum_horizon=1"
            "&maximum_horizon=24"
        )
    )

    summary_response = client.get(
        "/api/v1/forecast/summary"
    )

    alerts_response = client.get(
        "/api/v1/alerts"
    )

    active_alerts_response = client.get(
        "/api/v1/alerts/active"
    )

    metadata_response = client.get(
        "/api/v1/metadata"
    )

    pipeline_response = client.get(
        "/api/v1/pipeline/status"
    )

    openapi_response = client.get(
        "/openapi.json"
    )

/home/riyan/Riyan/projects/pearls-aqi-predictor/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
2026-07-27 20:46:56,245 INFO app.api.main artifact_cache_warmed phase_6_run_id=20260727T100418Z_aqi_ec8842b5 forecast_rows=72
2026-07-27 20:46:56,269 INFO app.api.middleware request_completed request_id=a169c4f2-d231-4084-9b9b-d30eacd9209d method=GET route=/api/v1/health/live status=200 duration_ms=10.668
2026-07-27 20:46:56,275 INFO httpx HTTP Request: GET http://testserver/api/v1/health/live "HTTP/1.1 200 OK"
2026-07-27 20:46:56,296 INFO app.api.middleware request_completed request_id=95cf84cd-adb8-4aaa-af2d-295fe724b4f5 method=GET route=/api/v1/health/ready status=200 duration_ms=9.572
2026-07-27 20:46:56,299 INFO httpx HTTP Request: GET http://testserver/api/v1/health/ready "HTTP/1.1 200 OK"
2026-07

### Review endpoint status codes

The current forecast may become stale depending on when the notebook is run.

Liveness must remain successful. Data endpoints should succeed when artifacts
are structurally valid. Readiness may return `503` when the configured
staleness threshold has been exceeded.

In [39]:
endpoint_status_df = pd.DataFrame(
    [
        {
            "endpoint": "/health/live",
            "status_code": (
                live_response.status_code
            ),
        },
        {
            "endpoint": "/health/ready",
            "status_code": (
                ready_response.status_code
            ),
        },
        {
            "endpoint": "/forecast",
            "status_code": (
                forecast_response.status_code
            ),
        },
        {
            "endpoint": "/forecast/hourly",
            "status_code": (
                hourly_response.status_code
            ),
        },
        {
            "endpoint": "/forecast/summary",
            "status_code": (
                summary_response.status_code
            ),
        },
        {
            "endpoint": "/alerts",
            "status_code": (
                alerts_response.status_code
            ),
        },
        {
            "endpoint": "/alerts/active",
            "status_code": (
                active_alerts_response
                .status_code
            ),
        },
        {
            "endpoint": "/metadata",
            "status_code": (
                metadata_response.status_code
            ),
        },
        {
            "endpoint": "/pipeline/status",
            "status_code": (
                pipeline_response.status_code
            ),
        },
        {
            "endpoint": "/openapi.json",
            "status_code": (
                openapi_response.status_code
            ),
        },
    ]
)

display(endpoint_status_df)

,endpoint,status_code
0,/health/live,200
1,/health/ready,200
2,/forecast,200
3,/forecast/hourly,200
4,/forecast/summary,200
5,/alerts,200
6,/alerts/active,200
7,/metadata,200
8,/pipeline/status,200
9,/openapi.json,200


### Validate the complete forecast response

The main forecast endpoint must return one stable public package containing all
72 ordered hourly records.

In [40]:
assert forecast_response.status_code == 200

forecast_payload = (
    forecast_response.json()
)

forecast_validation_summary = {
    "pipeline_run_id": (
        forecast_payload[
            "pipeline_run_id"
        ]
    ),
    "hourly_rows": len(
        forecast_payload[
            "hourly_forecast"
        ]
    ),
    "first_horizon": (
        forecast_payload[
            "hourly_forecast"
        ][0][
            "forecast_horizon_hours"
        ]
    ),
    "last_horizon": (
        forecast_payload[
            "hourly_forecast"
        ][-1][
            "forecast_horizon_hours"
        ]
    ),
    "active_alert_count": (
        forecast_payload[
            "active_alert_count"
        ]
    ),
    "freshness_status": (
        forecast_payload[
            "freshness"
        ]["status"]
    ),
}

display(
    pd.Series(
        forecast_validation_summary,
        name="value",
    ).to_frame()
)

assert (
    forecast_validation_summary[
        "hourly_rows"
    ]
    == 72
)

assert (
    forecast_validation_summary[
        "first_horizon"
    ]
    == 1
)

assert (
    forecast_validation_summary[
        "last_horizon"
    ]
    == 72
)

,value
pipeline_run_id,20260727T100418Z_aqi_ec8842b5
hourly_rows,72
first_horizon,1
last_horizon,72
active_alert_count,0
freshness_status,FRESH


### Validate hourly filtering

The requested horizon range should return only horizons 1 through 24.

In [41]:
assert hourly_response.status_code == 200

hourly_payload = hourly_response.json()

hourly_filter_validation = {
    "result_count": (
        hourly_payload["result_count"]
    ),
    "minimum_horizon": (
        hourly_payload[
            "applied_filters"
        ]["minimum_horizon"]
    ),
    "maximum_horizon": (
        hourly_payload[
            "applied_filters"
        ]["maximum_horizon"]
    ),
    "first_result_horizon": (
        hourly_payload[
            "records"
        ][0][
            "forecast_horizon_hours"
        ]
    ),
    "last_result_horizon": (
        hourly_payload[
            "records"
        ][-1][
            "forecast_horizon_hours"
        ]
    ),
}

display(
    pd.Series(
        hourly_filter_validation,
        name="value",
    ).to_frame()
)

assert (
    hourly_filter_validation[
        "result_count"
    ]
    == 24
)

,value
result_count,24
minimum_horizon,1
maximum_horizon,24
first_result_horizon,1
last_result_horizon,24


### Validate the no-alert response

The current forecast contains no active alert episodes. The alert endpoint must
return a valid empty collection rather than `null` or an error.

In [42]:
assert alerts_response.status_code == 200

alerts_payload = alerts_response.json()

alert_validation_summary = {
    "episode_count": (
        alerts_payload[
            "episode_count"
        ]
    ),
    "episodes_is_list": isinstance(
        alerts_payload["episodes"],
        list,
    ),
    "current_active_count": (
        active_alerts_response.json()[
            "current_count"
        ]
    ),
    "upcoming_count": (
        active_alerts_response.json()[
            "upcoming_count"
        ]
    ),
}

display(
    pd.Series(
        alert_validation_summary,
        name="value",
    ).to_frame()
)

assert (
    alert_validation_summary[
        "episode_count"
    ]
    == 0
)

assert (
    alert_validation_summary[
        "episodes_is_list"
    ]
)

,value
episode_count,0
episodes_is_list,True
current_active_count,0
upcoming_count,0


### Confirm middleware headers

Every API response should include a request ID. The processing-time header
supports lightweight operational diagnostics.

In [43]:
middleware_header_summary = {
    "request_id": (
        forecast_response.headers.get(
            "X-Request-ID"
        )
    ),
    "process_time_ms": (
        forecast_response.headers.get(
            "X-Process-Time-Ms"
        )
    ),
}

display(
    pd.Series(
        middleware_header_summary,
        name="value",
    ).to_frame()
)

assert (
    middleware_header_summary[
        "request_id"
    ]
)

assert (
    middleware_header_summary[
        "process_time_ms"
    ]
)

,value
request_id,60a643a9-c717-4277-afa0-ff592ef9e2f5
process_time_ms,108.829


### Confirm OpenAPI documentation

FastAPI generates the OpenAPI contract directly from the route definitions and
Pydantic response models.

In [44]:
assert openapi_response.status_code == 200

openapi_payload = openapi_response.json()

openapi_validation_summary = {
    "title": openapi_payload[
        "info"
    ]["title"],
    "version": openapi_payload[
        "info"
    ]["version"],
    "documented_paths": len(
        openapi_payload["paths"]
    ),
    "forecast_documented": (
        "/api/v1/forecast"
        in openapi_payload["paths"]
    ),
    "readiness_documented": (
        "/api/v1/health/ready"
        in openapi_payload["paths"]
    ),
}

display(
    pd.Series(
        openapi_validation_summary,
        name="value",
    ).to_frame()
)

,value
title,Pearls AQI Predictor API
version,1.0.0
documented_paths,9
forecast_documented,True
readiness_documented,True
